# Doctor Notes — DIAGNOSIS Column Cleaning

**Pipeline (in priority order):**
1. Load ICD-10 reference table from Snowflake
2. Parse & clean the DIAGNOSIS column
3. Extract ICD-10 codes where present → resolve canonical name directly
4. Exact match remaining terms against ICD-10 reference
5. Fuzzy match unresolved terms (misspellings / abbreviations)
6. BioBERT NER on genuine free text that still has no match
7. Write results back to Snowflake

## Cell 1 — Imports

In [1]:
import sys
import re
import ast
import json
from pathlib import Path

import pandas as pd
from rapidfuzz import process, fuzz
from transformers import pipeline
from tqdm import tqdm

_nb_file = globals().get("__vsc_ipynb_file__") or str(
    Path.cwd() / "preprocessing" / "doctor_notes_cleaning.ipynb"
)
_dashboards_root = Path(_nb_file).parent.parent / "analytics_app" / "dashboards"
sys.path.insert(0, str(_dashboards_root))

from snowflake_service.snowflake_client import SnowflakeClient

def get_connection():
    return SnowflakeClient()

c:\Users\Mercy\Documents\Clients\Snowflake Pipeline\xanalife\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cell 2 — Load ICD-10 Reference Table

This is the single source of truth for canonical names throughout the pipeline.
Every term we output must resolve to a name that exists in this table.

In [ ]:
icd10_sql = """
SELECT DISTINCT
    icd10_variation_code AS ICD10_CODE,
    icd10_variation_name AS ICD10_NAME
FROM HOSPITALS.STAGING.STG_UNIFIED_ICD10
WHERE icd10_variation_name IS NOT NULL
  AND icd10_variation_code IS NOT NULL
"""

icd10_ref = get_connection().query(icd10_sql)

# code → canonical name lookup  (e.g. "K29.3" → "Chronic superficial gastritis")
code_to_name = dict(zip(
    icd10_ref["ICD10_CODE"].str.strip(),
    icd10_ref["ICD10_NAME"].str.strip()
))

# lowercase name → canonical name lookup  (e.g. "gastritis" → best ICD-10 match)
# Used for exact string matching in Stage 4
icd10_names_lower = {
    name.lower().strip(): name.strip()
    for name in icd10_ref["ICD10_NAME"].dropna().unique()
}

# Plain list of canonical names for fuzzy matching in Stage 5
icd10_name_list = list(icd10_ref["ICD10_NAME"].dropna().unique())

print(f"ICD-10 reference loaded: {len(code_to_name):,} codes, {len(icd10_name_list):,} unique names")

## Cell 3 — Load Doctor Notes

In [ ]:
notes_sql = """
SELECT
    SOURCE_SCHEMA,
    DOCTOR_NOTE_ID,
    VISIT_ID,
    DIAGNOSIS
FROM HOSPITALS.STAGING.STG_EVALUATION_DOCTOR_NOTES
"""

df = get_connection().query(notes_sql)
print(f"Rows loaded: {len(df):,}")
print(df["DIAGNOSIS"].value_counts().head(10))

## Cell 4 — Stage 1: Parse & Clean the DIAGNOSIS Column

Handles:
- JSON array strings: `["urti"]`, `["K29.3 Chronic gastritis"]`
- Plain free text: `Other sepsis (Dr. Bonnke Arunga)`
- Multi-entry strings: `["K30 Dyspepsia _ B37 Candidiasis _"]`
- Noise removal: single characters, standalone punctuation, doctor names in brackets

In [ ]:
# Patterns
DR_NAME_PATTERN    = re.compile(r'\(\s*Dr\..*?\)', re.IGNORECASE)   # (Dr. Bonnke Arunga)
SINGLE_CHAR        = re.compile(r'^[a-zA-Z]$')                       # x, n, m
NOISE_ONLY         = re.compile(r'^[^a-zA-Z0-9#?/\.]+$')            # standalone punctuation
TRAILING_JUNK      = re.compile(r'[_\s]+$')                          # trailing underscores/spaces


def parse_diagnosis(raw):
    """
    Parse one raw DIAGNOSIS value into a list of cleaned text fragments.
    Does NOT expand abbreviations or resolve ICD-10 — that happens in later stages.
    """
    if not isinstance(raw, str) or not raw.strip():
        return []

    # Remove doctor name annotations before any other processing
    raw = DR_NAME_PATTERN.sub('', raw)

    # Attempt JSON array parse
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            # Join all elements — handles ["R19.7 Diarrhea", " unspecified"] as one string
            combined = ' '.join(str(p) for p in parsed)
        else:
            combined = str(parsed)
    except (ValueError, SyntaxError):
        # Plain text fallback
        combined = raw

    # Split on delimiters that separate multiple diagnoses within one string
    # Handles: "K30 Dyspepsia _ B37 Candidiasis _" and newline-separated entries
    fragments = re.split(r'\s*[_\n]+\s*', combined)

    cleaned = []
    for fragment in fragments:
        fragment = TRAILING_JUNK.sub('', fragment).strip()

        if not fragment:
            continue
        if SINGLE_CHAR.match(fragment):
            continue
        if NOISE_ONLY.match(fragment):
            continue

        cleaned.append(fragment)

    return cleaned


df["PARSED"] = df["DIAGNOSIS"].apply(parse_diagnosis)

# Quick sense check
non_empty = df["PARSED"].apply(lambda x: len(x) > 0).sum()
print(f"Rows with parseable content: {non_empty:,} of {len(df):,}")
print("\nSample:")
for raw, parsed in zip(df["DIAGNOSIS"].head(10), df["PARSED"].head(10)):
    print(f"  RAW:    {raw}")
    print(f"  PARSED: {parsed}")
    print()

## Cell 5 — Stage 2: Extract ICD-10 Codes Where Present

Priority 1: if a code is embedded in the text, resolve it directly from the reference table.
Codes are validated against the reference table — anatomical references like T12, L1 will not match and are ignored.

In [ ]:
# ICD-10 code pattern: letter + 2 digits + optional decimal + optional digits
# The space in "Z34. 91" is handled by stripping spaces from candidate matches
ICD10_PATTERN = re.compile(r'\b([A-Z]\d{2})\.?\s?(\d{0,2})\b')


def extract_icd10_codes(fragments):
    """
    Given a list of cleaned text fragments, extract any ICD-10 codes
    and resolve them to canonical names via the reference table.

    Returns:
        resolved  : list of canonical ICD-10 names found via code
        remainder : list of fragments with no valid code, passed to next stage
    """
    resolved  = []
    remainder = []

    for fragment in fragments:
        matches = ICD10_PATTERN.findall(fragment)
        found_valid_code = False

        for prefix, suffix in matches:
            # Reconstruct candidate codes — with and without decimal suffix
            candidates = [prefix]
            if suffix:
                candidates.append(f"{prefix}.{suffix}")

            for code in candidates:
                if code in code_to_name:
                    resolved.append(code_to_name[code])
                    found_valid_code = True
                    break  # Use first valid match per fragment

        if not found_valid_code:
            remainder.append(fragment)

    return resolved, remainder


df[["STAGE2_RESOLVED", "STAGE2_REMAINDER"]] = df["PARSED"].apply(
    lambda frags: pd.Series(extract_icd10_codes(frags))
)

code_hits = df["STAGE2_RESOLVED"].apply(lambda x: len(x) > 0).sum()
print(f"Rows resolved via ICD-10 code: {code_hits:,}")
print(f"Rows with remainder to process: {df['STAGE2_REMAINDER'].apply(lambda x: len(x) > 0).sum():,}")

## Cell 6 — Stage 3: Exact Match Against ICD-10 Name List

For fragments that had no embedded code, try exact (case-insensitive) match
against ICD-10 canonical names. `gastritis` → `Gastritis, unspecified`.

In [ ]:
def exact_match(fragments):
    """
    Try exact case-insensitive match of each fragment against ICD-10 name list.

    Returns:
        resolved  : canonical names found via exact match
        remainder : fragments with no exact match, passed to fuzzy stage
    """
    resolved  = []
    remainder = []

    for fragment in fragments:
        key = fragment.lower().strip()
        if key in icd10_names_lower:
            resolved.append(icd10_names_lower[key])
        else:
            remainder.append(fragment)

    return resolved, remainder


df[["STAGE3_RESOLVED", "STAGE3_REMAINDER"]] = df["STAGE2_REMAINDER"].apply(
    lambda frags: pd.Series(exact_match(frags))
)

exact_hits = df["STAGE3_RESOLVED"].apply(lambda x: len(x) > 0).sum()
print(f"Rows resolved via exact match: {exact_hits:,}")
print(f"Rows with remainder to process: {df['STAGE3_REMAINDER'].apply(lambda x: len(x) > 0).sum():,}")

# Inspect what exact match is catching
print("\nSample exact matches:")
sample = df[df["STAGE3_RESOLVED"].apply(lambda x: len(x) > 0)]
for _, row in sample.head(5).iterrows():
    print(f"  INPUT:    {row['STAGE2_REMAINDER']}")
    print(f"  RESOLVED: {row['STAGE3_RESOLVED']}")
    print()

## Cell 7 — Stage 4: Fuzzy Match (Misspellings & Abbreviations)

For fragments that had no exact match, use RapidFuzz to find the closest
ICD-10 canonical name. A confidence threshold of 80 is used — tune this
based on the sample output below before running on the full dataset.

In [ ]:
FUZZY_THRESHOLD = 80  # 0-100. Lower = more matches but more false positives.


def fuzzy_match(fragments):
    """
    Use RapidFuzz to match each fragment against the ICD-10 name list.
    Only accepts matches above FUZZY_THRESHOLD confidence.

    Returns:
        resolved  : canonical names found via fuzzy match
        remainder : fragments below threshold, passed to BioBERT
    """
    resolved  = []
    remainder = []

    for fragment in fragments:
        # token_sort_ratio handles word order differences
        # e.g. "hypertension essential" matches "Essential hypertension"
        result = process.extractOne(
            fragment,
            icd10_name_list,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=FUZZY_THRESHOLD
        )
        if result:
            resolved.append(result[0])  # result[0] is the canonical name
        else:
            remainder.append(fragment)

    return resolved, remainder


# Run on the remainder from Stage 3
has_remainder = df["STAGE3_REMAINDER"].apply(lambda x: len(x) > 0)
remainder_df  = df[has_remainder].copy()

print(f"Running fuzzy match on {len(remainder_df):,} rows...")
remainder_df[["STAGE4_RESOLVED", "STAGE4_REMAINDER"]] = remainder_df["STAGE3_REMAINDER"].apply(
    lambda frags: pd.Series(fuzzy_match(frags))
)

df = df.join(remainder_df[["STAGE4_RESOLVED", "STAGE4_REMAINDER"]])
df["STAGE4_RESOLVED"]  = df["STAGE4_RESOLVED"].apply(lambda x: x if isinstance(x, list) else [])
df["STAGE4_REMAINDER"] = df["STAGE4_REMAINDER"].apply(lambda x: x if isinstance(x, list) else [])

fuzzy_hits = df["STAGE4_RESOLVED"].apply(lambda x: len(x) > 0).sum()
print(f"Rows resolved via fuzzy match: {fuzzy_hits:,}")
print(f"Rows with remainder for BioBERT: {df['STAGE4_REMAINDER'].apply(lambda x: len(x) > 0).sum():,}")

# IMPORTANT: Inspect fuzzy matches before proceeding
# Check for false positives — if confidence is too low, raise FUZZY_THRESHOLD
print("\nSample fuzzy matches (verify these look correct):")
sample = remainder_df[remainder_df["STAGE4_RESOLVED"].apply(lambda x: len(x) > 0)]
for _, row in sample.head(10).iterrows():
    print(f"  INPUT:    {row['STAGE3_REMAINDER']}")
    print(f"  RESOLVED: {row['STAGE4_RESOLVED']}")
    print()

## Cell 8 — Stage 5: BioBERT NER on Genuine Free Text

Only runs on rows that could not be resolved by any earlier stage.
These are rows with genuine unstructured prose — not single terms or abbreviations.
BioBERT extracts disease entities, which are then fuzzy-matched back to ICD-10 names.

In [ ]:
biobert_df = df[df["STAGE4_REMAINDER"].apply(lambda x: len(x) > 0)].copy()
print(f"Rows reaching BioBERT: {len(biobert_df):,}")

if len(biobert_df) == 0:
    print("No rows need BioBERT — all resolved by earlier stages.")
else:
    print("Loading BioBERT...")
    biobert_ner = pipeline(
        "ner",
        model="alvaroalon2/biobert_diseases_ner",
        aggregation_strategy="simple",
        device=-1  # Change to 0 if GPU available
    )

    def run_biobert_on_fragments(fragments, batch_size=64):
        """
        Run BioBERT on a list of text fragments in batches.
        Extracted entities are then fuzzy-matched back to ICD-10 canonical names.
        """
        if not fragments:
            return [], []

        # BioBERT inference in one batched call
        batch_results = biobert_ner(fragments, batch_size=batch_size)

        resolved  = []
        remainder = []

        for fragment, entities in zip(fragments, batch_results):
            terms = list({e["word"].strip().lower() for e in entities})

            if not terms:
                remainder.append(fragment)
                continue

            for term in terms:
                result = process.extractOne(
                    term,
                    icd10_name_list,
                    scorer=fuzz.token_sort_ratio,
                    score_cutoff=FUZZY_THRESHOLD
                )
                if result:
                    resolved.append(result[0])
                else:
                    remainder.append(term)

        return resolved, remainder

    biobert_df[["STAGE5_RESOLVED", "STAGE5_REMAINDER"]] = biobert_df["STAGE4_REMAINDER"].apply(
        lambda frags: pd.Series(run_biobert_on_fragments(frags))
    )

    df = df.join(biobert_df[["STAGE5_RESOLVED", "STAGE5_REMAINDER"]])

df["STAGE5_RESOLVED"]  = df["STAGE5_RESOLVED"].apply(lambda x: x if isinstance(x, list) else [])
df["STAGE5_REMAINDER"] = df["STAGE5_REMAINDER"].apply(lambda x: x if isinstance(x, list) else [])

biobert_hits = df["STAGE5_RESOLVED"].apply(lambda x: len(x) > 0).sum()
print(f"Rows resolved via BioBERT: {biobert_hits:,}")
print(f"Unresolved after all stages: {df['STAGE5_REMAINDER'].apply(lambda x: len(x) > 0).sum():,}")

## Cell 9 — Consolidate Results

Merge resolved names from all stages into a single output column.
Each row gets a list of canonical ICD-10 names and the stage that resolved them.

In [ ]:
def consolidate(row):
    """
    Merge all resolved names from every stage into one deduplicated list.
    Also returns which stage resolved the row (for audit/debugging).
    """
    all_resolved = (
        row["STAGE2_RESOLVED"] +
        row["STAGE3_RESOLVED"] +
        row["STAGE4_RESOLVED"] +
        row["STAGE5_RESOLVED"]
    )

    # Deduplicate while preserving order
    seen = set()
    unique = []
    for name in all_resolved:
        if name not in seen:
            seen.add(name)
            unique.append(name)

    # Which stage first produced a result
    if row["STAGE2_RESOLVED"]:
        stage = "ICD10_CODE"
    elif row["STAGE3_RESOLVED"]:
        stage = "EXACT_MATCH"
    elif row["STAGE4_RESOLVED"]:
        stage = "FUZZY_MATCH"
    elif row["STAGE5_RESOLVED"]:
        stage = "BIOBERT"
    elif row["PARSED"] == []:
        stage = "EMPTY_INPUT"
    else:
        stage = "UNRESOLVED"

    return pd.Series({
        "CANONICAL_NAMES": unique,
        "RESOLUTION_STAGE": stage,
        "IS_RESOLVED": len(unique) > 0
    })


df[["CANONICAL_NAMES", "RESOLUTION_STAGE", "IS_RESOLVED"]] = df.apply(consolidate, axis=1)

# Summary
print("Resolution breakdown:")
print(df["RESOLUTION_STAGE"].value_counts())
print(f"\nTotal resolved: {df['IS_RESOLVED'].sum():,} of {len(df):,}")
print(f"Resolution rate: {df['IS_RESOLVED'].mean():.1%}")

## Cell 10 — Sense Check Output

Review before writing back to Snowflake.
Pay particular attention to FUZZY_MATCH rows — false positives are most likely there.

In [ ]:
# Sample of each resolution stage
for stage in ["ICD10_CODE", "EXACT_MATCH", "FUZZY_MATCH", "BIOBERT", "UNRESOLVED"]:
    sample = df[df["RESOLUTION_STAGE"] == stage][["DIAGNOSIS", "CANONICAL_NAMES"]].head(5)
    if len(sample):
        print(f"\n── {stage} ──")
        for _, row in sample.iterrows():
            print(f"  RAW:    {row['DIAGNOSIS']}")
            print(f"  OUTPUT: {row['CANONICAL_NAMES']}")
            print()

## Cell 11 — Write Results to Snowflake

Output table: `HOSPITALS.STAGING.STG_DIAGNOSIS_CLEANED`

One row per doctor note. `CANONICAL_NAMES` is stored as a JSON array string
so it can be unpivoted downstream when mapping to `fact_diagnoses`.

In [ ]:
output = df[[
    "SOURCE_SCHEMA",
    "DOCTOR_NOTE_ID",
    "VISIT_ID",
    "DIAGNOSIS",
    "CANONICAL_NAMES",
    "RESOLUTION_STAGE",
    "IS_RESOLVED"
]].copy()

# Serialise list to JSON string for Snowflake storage
output["CANONICAL_NAMES"] = output["CANONICAL_NAMES"].apply(json.dumps)
output["IS_RESOLVED"]     = output["IS_RESOLVED"].astype(int)

get_connection().write(
    df=output,
    table_name="STG_DIAGNOSIS_CLEANED",
    schema="STAGING",
    database="HOSPITALS",
    mode="overwrite"
)

print(f"Written {len(output):,} rows to HOSPITALS.STAGING.STG_DIAGNOSIS_CLEANED")